# Workshop-Notebook: Grundlagen der KI-Agenten & Werkzeugnutzung

Dieses Notebook zeigt Ihnen Schritt für Schritt, wie Sie ein Sprachmodell (**Gemini**) mit einem **Werkzeug (Tool)** verbinden und eine einfache **ReAct-Schleife (Orchestrierung)** aufbauen.

**Event:** Digitaltag 2026 | *Mittelstand-Digital Zentrum Spreeland*

### Schritt 1: Installation & Setup
Zuerst installieren wir das offizielle Google GenAI SDK und konfigurieren unseren API-Schlüssel.

**API-Key einrichten:**
Sie benötigen einen kostenlosen Gemini API-Schlüssel von [Google AI Studio](https://aistudio.google.com/app/api-keys).
In Google Colab gibt es zwei Möglichkeiten, den Schlüssel sicher zu hinterlegen:
1. **Interaktiv (Empfohlen für Präsentationen):** Wenn Sie die Zelle unten ausführen, werden Sie nach Ihrem Schlüssel gefragt. Die Eingabe wird maskiert.
2. **Colab-Geheimnisse (Secrets):** Klicken Sie links auf das Schlüssel-Symbol (🔑), fügen Sie ein neues Geheimnis namens `GEMINI_API_KEY` hinzu, fügen Sie Ihren API-Key ein und aktivieren Sie den Schalter "Notebook-Zugriff".

In [ ]:
# Installation der SDKs und Upgrade von Protobuf zur Vermeidung von Inkompatibilitäten
%pip install --upgrade protobuf google-genai google-antigravity

In [ ]:
import os
import getpass
from google import genai
from google.genai import types
from google.genai.errors import APIError

# 1. Versuche den Key aus den Colab-Secrets zu laden (und whitespaces zu entfernen)
try:
    from google.colab import userdata
    key = userdata.get("GEMINI_API_KEY")
    if key:
        os.environ["GEMINI_API_KEY"] = key.strip()
except Exception:
    pass

# 2. Falls nicht gesetzt, frage den Nutzer interaktiv nach dem Key
if not os.environ.get("GEMINI_API_KEY"):
    key = getpass.getpass("Bitte fügen Sie Ihren Gemini API-Key ein und drücken Sie Enter: ")
    os.environ["GEMINI_API_KEY"] = key.strip()

# 3. Initialisierung des GenAI Clients
client = genai.Client()
model_id = 'gemini-2.5-flash'
print("Gemini API-Schlüssel wurde erfolgreich konfiguriert und der Client initialisiert!")

### Schritt 2: Einfache Modellanfrage
Bevor wir mit Werkzeugen (Tools) arbeiten, testen wir, ob wir das Sprachmodell direkt anfragen können. Wir stellen eine einfache Frage wie: *"Was ist ein KI-Agent?"* und formatieren die Antwort für bessere Lesbarkeit.

In [ ]:
import textwrap

# Einfache Anfrage an Gemini senden
response = client.models.generate_content(
    model=model_id,
    contents="Was ist ein KI-Agent? Beantworten Sie in 2 bis 3 kurzen Sätzen."
)

print("Antwort von Gemini:\n")
# Formatierter Zeilenumbruch für bessere Lesbarkeit im Notebook
print(textwrap.fill(response.text, width=80))

### Schritt 3: Echte API als Werkzeug definieren (Tool)
Nun wollen wir dem Modell Zugriff auf Echtzeit-Informationen geben. Wir definieren eine Python-Funktion, die die **freie Open-Meteo-Wetter-API** abfragt, um aktuelle Wetterdaten für Cottbus zu ermitteln (ohne API-Key).

In [ ]:
import urllib.request
import json

def get_wetter_cottbus(datum: str) -> str:
    """Fragt das aktuelle Wetter für Cottbus an einem bestimmten Datum ab.
    
    Args:
        datum: Das gewünschte Datum im Format JJJJ-MM-TT (z. B. 2026-06-26 oder heute/morgen)
    """
    # Cottbus Koordinaten
    lat = 51.7563
    lon = 14.3329
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&daily=temperature_2m_max,temperature_2m_min,precipitation_probability_max&timezone=Europe/Berlin"
    
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response:
            data = json.loads(response.read().decode())
        
        daily = data.get("daily", {})
        time_list = daily.get("time", [])
        
        if datum in time_list:
            idx = time_list.index(datum)
            temp_max = daily.get("temperature_2m_max", [])[idx]
            temp_min = daily.get("temperature_2m_min", [])[idx]
            rain_prob = daily.get("precipitation_probability_max", [])[idx]
            return (f"Wettervorhersage für Cottbus am {datum}: "
                    f"Höchsttemperatur {temp_max}°C, Tiefsttemperatur {temp_min}°C, "
                    f"Regenwahrscheinlichkeit {rain_prob}%.")
        else:
            return (f"Für das Datum {datum} liegen leider keine aktuellen 7-Tage-Vorhersagedaten vor. "
                    f"Bitte wählen Sie ein Datum im Bereich {time_list[0]} bis {time_list[-1]}.")
    except Exception as e:
        return f"Fehler bei der Wetterabfrage: {str(e)}. (Ersatzwert: 20 Grad, leicht bewölkt)."

### Schritt 4: Modell mit Werkzeug verbinden (Tool Binding)
Wir senden eine Anfrage, die Wetterdaten benötigt. Wir generieren das Datum für **morgen** dynamisch, damit die Abfrage immer in der 7-Tage-Vorhersage der API liegt.

In [ ]:
response = client.models.generate_content(
    model=model_id,
    contents=prompt,
    config=types.GenerateContentConfig(
        # Hier registrieren wir unsere Funktion als Werkzeug
        tools=[get_wetter_cottbus],
        temperature=0,
        # Wir deaktivieren das automatische Ausführen, um den rohen Funktionsaufruf zu sehen
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)
    )
)

# Das Modell führt die Funktion nicht selbst aus, sondern fordert uns dazu auf:
print("Funktionsaufrufe des Modells:", response.function_calls)

### Schritt 5: Die Orchestrierungsschleife (ReAct Loop)
Mit `automatic_function_calling` überlassen wir dem SDK die Ausführung der Funktion im Hintergrund. Der Agent holt sich die Wetterdaten von der API und formuliert die finale Antwort, die wir wieder passend umbrechen.

In [ ]:
import textwrap
from datetime import datetime, timedelta

morgen = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")
prompt = f"Brauche ich am {morgen} in Cottbus eine Regenjacke? Beantworte basierend auf dem Wetterbericht."

response_auto = client.models.generate_content(
    model=model_id,
    contents=prompt,
    config=types.GenerateContentConfig(
        tools=[get_wetter_cottbus],
        automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=3)
    )
)

print("Endgültige Antwort des Agenten:\n")
print(textwrap.fill(response_auto.text, width=80))

### Schritt 6: Höhere Abstraktion mit dem Google Antigravity SDK (Optional)
Für komplexere Agenten (z. B. mit autonomem Dateizugriff, Shell-Ausführung, Richtlinien oder Multi-Agenten-Orchestrierung) bietet Google das **Google Antigravity SDK** (`google-antigravity`) an.

Hier installieren wir das SDK und bauen einen einfachen Agenten, der in einer sicheren Umgebung (mit einer Richtlinie, die alle Aktionen erlaubt) ausgeführt wird.

In [ ]:
import asyncio
import os
from google.antigravity import Agent, LocalAgentConfig
from google.antigravity.hooks import policy

# 1. Agenten konfigurieren
# Wir erlauben dem Agenten Werkzeuge autonom auszuführen (policy.allow_all()).
config = LocalAgentConfig(
    system_instructions="Du bist ein hilfreicher KI-Assistent.",
    policies=[policy.allow_all()],
    api_key=os.environ.get("GEMINI_API_KEY")
)

# 2. Den Agenten in einer asynchronen Schleife ausführen
async def run_agent():
    async with Agent(config) as agent:
        response = await agent.chat("Schreibe ein kurzes Gedicht über Cottbus und die Lausitz.")
        text_content = await response.text()
        print("Antwort des Agenten:\n")
        print(text_content)

# In Jupyter / Colab läuft bereits eine Event-Loop.
# Daher führen wir die asynchrone Funktion direkt mit `await` aus:
await run_agent()